# Stage 3 — Abnormal Returns & Downside Labels

Computes cumulative abnormal returns (CAR) for each filing and assigns a binary downside label.

**Input** : `data/filings_with_text.csv`  
**Output** : `data/filings_labeled.csv` — full dataset with `car`, `event_date`, and `downside` columns

**Event window** : CAR[0, +1] — market-adjusted model (stock return minus SPY return)  
**Downside threshold** : CAR < −2%  
**Robustness windows** : CAR[0, 0] and CAR[−1, +1] computed alongside for later use

## 0. Install dependencies

## 1. Imports and configuration

In [ ]:
# importing the libraries 
import pandas as pd
import numpy as np
import yfinance as yf
import pytz
import pandas_market_calendars as mcal
from pathlib import Path
from datetime import timedelta
import time

In [ ]:
# setting up the paths 
DATA_DIR     = Path("data")
INPUT_PATH   = DATA_DIR / "filings_with_text.csv"
OUTPUT_PATH  = DATA_DIR / "filings_labeled.csv"

# event study parameters
MARKET_CLOSE_ET = 16                # NYSE closes at 4pm ET
DOWNSIDE_THRESHOLD = -0.02          # CAR < -2% = downside event
ET_ZONE = pytz.timezone("America/New_York")
UTC_ZONE = pytz.utc

# price data parameters
PRICE_START = "2023-01-01"          # extra buffer before 2023
PRICE_END   = "2026-04-30"          # extra buffer after 2026

print("Configuration loaded.")

## 2. Load filings

In [ ]:
df = pd.read_csv(INPUT_PATH)
print(f"Loaded {len(df)} filings")
print(f"Columns: {list(df.columns)}")
print(f"\nSample filingDatetime values:")
print(df['filingDatetime'].head(5).tolist())

## 3. Build NYSE trading calendar

We need a list of valid NYSE trading days to:
- Roll after-hours filings to the next trading day
- Look up day +1 without accidentally landing on a weekend or holiday

In [ ]:
nyse = mcal.get_calendar("NYSE")
schedule = nyse.schedule(start_date=PRICE_START, end_date=PRICE_END)
trading_days = pd.DatetimeIndex(schedule.index.date).normalize()

def next_trading_day(date: pd.Timestamp) -> pd.Timestamp:
    """Returns the next NYSE trading day after the given date."""
    future = trading_days[trading_days > date]
    return future[0] if len(future) > 0 else None

def prev_trading_day(date: pd.Timestamp) -> pd.Timestamp:
    """Returns the previous NYSE trading day before the given date."""
    past = trading_days[trading_days < date]
    return past[-1] if len(past) > 0 else None

def nearest_trading_day(date: pd.Timestamp) -> pd.Timestamp:
    """Returns the date itself if a trading day, else the next trading day."""
    if date in trading_days:
        return date
    return next_trading_day(date)

print(f"Trading calendar built: {len(trading_days)} trading days ({PRICE_START} to {PRICE_END})")
print(f"Sample: {list(trading_days[:5])}")

## 4. Assign event dates

Core logic:
- Parse `filingDatetime` (stored as UTC by EDGAR)
- Convert to ET
- If filing time (ET) is before 4pm → event day 0 = filing date
- If filing time (ET) is at/after 4pm → event day 0 = next trading day
- Day −1 and day +1 are looked up from the NYSE calendar

In [ ]:
def assign_event_dates(row) -> pd.Series:
    """
    Given a filingDatetime string, returns:
        event_date   : day 0 (first day market can react)
        event_date_m1: day -1 (for robustness window)
        event_date_p1: day +1 (second day of main window)
    """
    dt_str = row["filingDatetime"]

    try:
        # Parse datetime — EDGAR stores as UTC
        dt_utc = pd.Timestamp(dt_str).tz_localize(UTC_ZONE) if pd.Timestamp(dt_str).tzinfo is None \
                 else pd.Timestamp(dt_str).tz_convert(UTC_ZONE)

        # Convert to ET
        dt_et = dt_utc.tz_convert(ET_ZONE)

        # Filing date in ET (date only, no time)
        filing_date = pd.Timestamp(dt_et.date())

        # Determine day 0
        if dt_et.hour < MARKET_CLOSE_ET:
            # Filed during market hours — day 0 is the filing date
            # (but make sure it's actually a trading day)
            day0 = nearest_trading_day(filing_date)
        else:
            # Filed after market close — day 0 is the next trading day
            day0 = next_trading_day(filing_date)

        if day0 is None:
            return pd.Series({"event_date": None, "event_date_m1": None, "event_date_p1": None})

        day_m1 = prev_trading_day(day0)
        day_p1 = next_trading_day(day0)

        return pd.Series({
            "event_date":    day0,
            "event_date_m1": day_m1,
            "event_date_p1": day_p1,
        })

    except Exception as e:
        return pd.Series({"event_date": None, "event_date_m1": None, "event_date_p1": None})


print("Assigning event dates...")
event_dates = df.apply(assign_event_dates, axis=1)
df = pd.concat([df, event_dates], axis=1)

# Drop filings where event date could not be determined
before = len(df)
df = df.dropna(subset=["event_date", "event_date_p1"]).reset_index(drop=True)
print(f"Dropped {before - len(df)} filings with missing event dates")
print(f"Remaining: {len(df)} filings")

# Quick sanity check
print("\nSample event date assignments:")
print(df[["ticker", "filingDatetime", "event_date", "event_date_m1", "event_date_p1"]].head(8).to_string())

## 5. Download price data from Yahoo Finance

Single bulk download for all tickers + SPY. Much faster than one request per filing.

In [ ]:
tickers = sorted(df["ticker"].unique().tolist())
tickers_with_spy = tickers + ["SPY"]

CACHE_FILE = Path("data/prices_cache.csv")

if CACHE_FILE.exists():
    cached = pd.read_csv(CACHE_FILE, index_col=0, parse_dates=True)
    done = set(cached.columns.tolist())
    print(f"Resuming — {len(done)} tickers already cached")
else:
    cached = pd.DataFrame()
    done = set()

remaining = [t for t in tickers_with_spy if t not in done]
print(f"Remaining: {len(remaining)} tickers")

for i, ticker in enumerate(remaining):
    try:
        data = yf.Ticker(ticker).history(
            start=PRICE_START,
            end=PRICE_END,
            auto_adjust=True,
        )["Close"]

        data.index = pd.to_datetime(data.index).tz_localize(None).normalize()

        if len(data) > 0:
            cached[ticker] = data
            print(f"[{i+1}/{len(remaining)}] {ticker}: {len(data)} rows")
        else:
            print(f"[{i+1}/{len(remaining)}] {ticker}: no data")
    except Exception as e:
        print(f"[{i+1}/{len(remaining)}] {ticker}: failed — {e}")

    if (i + 1) % 20 == 0:
        cached.to_csv(CACHE_FILE)
        print(f"  Cache saved ({len(cached.columns)} tickers)")

    time.sleep(0.5)

cached.to_csv(CACHE_FILE)
raw_prices = cached
print(f"Price history starts: {raw_prices.index.min().date()}")
print(f"Price history ends  : {raw_prices.index.max().date()}")
print(f"SPY available       : {'SPY' in raw_prices.columns}")
print(f"\nDone. Price data shape: {raw_prices.shape}")

In [ ]:
# extend existing cache back to 2021-01-01 for SPY and all firm tickers
raw_prices = pd.read_csv(CACHE_FILE, index_col=0, parse_dates=True)

tickers_to_extend = raw_prices.columns.tolist()
extension = yf.download(
    tickers_to_extend,
    start="2021-01-01",
    end="2021-11-30",
    auto_adjust=True,
    progress=True,
)["Close"]

extension.index = pd.to_datetime(extension.index).tz_localize(None).normalize()

# Merge extension with existing cache
raw_prices = pd.concat([extension, raw_prices]).sort_index()
raw_prices = raw_prices[~raw_prices.index.duplicated(keep="last")]
raw_prices.to_csv(CACHE_FILE)

print(f"Price history now starts: {raw_prices.index.min().date()}")
print(f"Price history now ends  : {raw_prices.index.max().date()}")

In [ ]:
def get_price(ticker: str, date: pd.Timestamp) -> float | None:
    """Returns the adjusted close price for a ticker on a given date."""
    if ticker not in raw_prices.columns:
        return None
    date = pd.Timestamp(date).normalize()
    if date not in raw_prices.index:
        return None
    val = raw_prices.loc[date, ticker]
    return float(val) if pd.notna(val) else None

def get_spy_price(date: pd.Timestamp) -> float | None:
    return get_price("SPY", date)

print("Fetching prices for t-1, t0, t+1...")
df["price_m1"]     = df.apply(lambda r: get_price(r["ticker"], r["event_date_m1"]), axis=1)
df["price_0"]      = df.apply(lambda r: get_price(r["ticker"], r["event_date"]),    axis=1)
df["price_p1"]     = df.apply(lambda r: get_price(r["ticker"], r["event_date_p1"]), axis=1)
df["spy_price_m1"] = df.apply(lambda r: get_spy_price(r["event_date_m1"]), axis=1)
df["spy_price_0"]  = df.apply(lambda r: get_spy_price(r["event_date"]),    axis=1)
df["spy_price_p1"] = df.apply(lambda r: get_spy_price(r["event_date_p1"]), axis=1)

# Drop rows where t0 or t+1 prices are missing
before = len(df)
df = df.dropna(subset=["price_0", "price_p1", "spy_price_0", "spy_price_p1"]).reset_index(drop=True)
print(f"Dropped {before - len(df)} filings with missing prices")
print(f"Remaining: {len(df)} filings")
print(df[["ticker", "event_date", "price_m1", "price_0", "price_p1"]].head(8).to_string())

### Section6: Market Model (MacKinlay 1997)

In [ ]:
# estimating firm-specific market model (MacKinlay 1997) - Estimation window: 200 trading days ending 20 days before event day 0

ESTIMATION_DAYS = 200
ESTIMATION_GAP  = 20

def estimate_market_model(ticker: str, event_date: pd.Timestamp):
    event_date = pd.Timestamp(event_date).normalize()

    try:
        day0_idx = trading_days.get_loc(event_date)
    except KeyError:
        return None, None

    end_idx   = day0_idx - ESTIMATION_GAP
    start_idx = end_idx  - ESTIMATION_DAYS

    if start_idx < 0 or end_idx <= start_idx:
        return None, None

    est_days = trading_days[start_idx:end_idx]

    if ticker not in raw_prices.columns or "SPY" not in raw_prices.columns:
        return None, None

    firm_prices = raw_prices.loc[raw_prices.index.isin(est_days), ticker].dropna()
    spy_prices  = raw_prices.loc[raw_prices.index.isin(est_days), "SPY"].dropna()

    common = firm_prices.index.intersection(spy_prices.index)
    if len(common) < 30:
        return None, None

    r_firm = firm_prices.loc[common].pct_change().dropna()
    r_spy  = spy_prices.loc[common].pct_change().dropna()
    common2 = r_firm.index.intersection(r_spy.index)

    if len(common2) < 30:
        return None, None

    r_f = r_firm.loc[common2].values
    r_m = r_spy.loc[common2].values

    X = np.column_stack([np.ones(len(r_m)), r_m])
    try:
        coeffs, _, _, _ = np.linalg.lstsq(X, r_f, rcond=None)
        alpha, beta = coeffs
        return float(alpha), float(beta)
    except Exception:
        return None, None


print("Estimating market model parameters...")
params = df.apply(
    lambda r: pd.Series(
        estimate_market_model(r["ticker"], r["event_date"]),
        index=["mm_alpha", "mm_beta"]
    ),
    axis=1,
)
df = pd.concat([df, params], axis=1)

n_estimated = df["mm_beta"].notna().sum()
n_fallback  = df["mm_beta"].isna().sum()
print(f"Market model estimated     : {n_estimated} filings")
print(f"Insufficient history       : {n_fallback} filings (falling back to beta=1, alpha=0)")

df["mm_alpha"] = df["mm_alpha"].fillna(0.0)
df["mm_beta"]  = df["mm_beta"].fillna(1.0)
df["mm_flag"]  = ((df["mm_beta"] == 1.0) & (df["mm_alpha"] == 0.0))

print(f"\nBeta distribution:")
print(df["mm_beta"].describe().round(4))

### Section 7: Abnormal Returns

In [ ]:
# computing abnormal returns using firm-specific market model
# CAR[0,+1] = AR_0 + AR_1
# AR_t = R_t - (alpha + beta * R_market_t)

# Day 0 returns
df["ret_0"]     = (df["price_0"]     - df["price_m1"])     / df["price_m1"]
df["ret_spy_0"] = (df["spy_price_0"] - df["spy_price_m1"]) / df["spy_price_m1"]

# Day +1 returns
df["ret_1"]     = (df["price_p1"]    - df["price_0"])      / df["price_0"]
df["ret_spy_1"] = (df["spy_price_p1"]- df["spy_price_0"])  / df["spy_price_0"]

# Abnormal returns per day
df["ar_0"] = df["ret_0"] - (df["mm_alpha"] + df["mm_beta"] * df["ret_spy_0"])
df["ar_1"] = df["ret_1"] - (df["mm_alpha"] + df["mm_beta"] * df["ret_spy_1"])

# CAR[0,+1] = sum of abnormal returns over event window
df["car_0_1"] = df["ar_0"] + df["ar_1"]

# Keep these for consistency with existing column names
df["ret_stock"]    = df["ret_1"]                          # day +1 stock return (kept for schema compatibility)
df["ret_spy"]      = df["ret_spy_1"]                      # day +1 SPY return
df["expected_ret"] = df["mm_alpha"] + df["mm_beta"] * df["ret_spy_1"]  # expected return day +1

# Market-adjusted CAR[0,+1] as robustness series
df["car_0_1_mktadj"] = (df["ret_0"] - df["ret_spy_0"]) + (df["ret_1"] - df["ret_spy_1"])

# Downside label — threshold set at 25th percentile of new CAR distribution
DOWNSIDE_THRESHOLD = df["car_0_1"].quantile(0.25)
df["downside"] = (df["car_0_1"] < DOWNSIDE_THRESHOLD).astype(int)

# dripping helper columns not needed in output
df.drop(columns=["ret_0", "ret_spy_0", "ret_1", "ret_spy_1", "ar_0", "ar_1"], inplace=True)

print("CAR[0,+1] distribution (market model):")
print(df["car_0_1"].describe(percentiles=[0.10, 0.25, 0.50]).round(4))
print(f"\nMarket-adjusted CAR[0,+1] distribution (robustness):")
print(df["car_0_1_mktadj"].describe(percentiles=[0.10, 0.25, 0.50]).round(4))
print(f"\nDownside threshold (25th percentile): {DOWNSIDE_THRESHOLD:.4f} ({DOWNSIDE_THRESHOLD*100:.2f}%)")
print(f"Downside rate: {df['downside'].mean()*100:.1f}%")
print(f"Downside count: {df['downside'].sum()}")
print(f"Fallback to market-adjusted (beta=1): {df['mm_flag'].sum()} filings")
print(f"\nTop 5 biggest drops:")
print(df.nsmallest(5, "car_0_1")[["ticker", "filingDate", "car_0_1", "mm_beta"]].to_string())

### Section 8: Empirical justification for downside threshold

In [ ]:
# Empirical justification for downside threshold

p25 = df["car_0_1"].quantile(0.25)
sd1 = df["car_0_1"].mean() - df["car_0_1"].std()

print(f"Bottom quartile (25th percentile) : {p25:.4f}")
print(f"Mean minus 1 SD                   : {sd1:.4f}")

print("\nRobustness checks across alternative thresholds:")
for thresh in [-0.02, -0.03, -0.04, round(p25, 4)]:
    n    = (df["car_0_1"] < thresh).sum()
    rate = (df["car_0_1"] < thresh).mean() * 100
    print(f"  CAR < {thresh*100:.1f}%  →  {n} filings  ({rate:.1f}% of sample)")

## 9. Save output

In [ ]:
cols_to_save = [
    "ticker", "cik", "accessionNumber", "filingDate", "filingDatetime",
    "event_date", "event_date_m1", "event_date_p1",
    "price_m1", "price_0", "price_p1",
    "spy_price_m1", "spy_price_0", "spy_price_p1",
    "mm_alpha", "mm_beta", "mm_flag",
    "ret_stock", "ret_spy", "expected_ret",
    "car_0_1", "car_0_1_mktadj", "downside",
]
df[cols_to_save].to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(df)} filings with prices → {OUTPUT_PATH}")

## 10. Validation and distribution check

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/filings_labeled.csv")

# ── 1. Recompute CAR from scratch using raw prices ────────────────────────────
df["ret_0"]     = (df["price_0"]     - df["price_m1"])     / df["price_m1"]
df["ret_1"]     = (df["price_p1"]    - df["price_0"])      / df["price_0"]
df["ret_spy_0"] = (df["spy_price_0"] - df["spy_price_m1"]) / df["spy_price_m1"]
df["ret_spy_1"] = (df["spy_price_p1"]- df["spy_price_0"])  / df["spy_price_0"]

df["ar_0_check"] = df["ret_0"] - (df["mm_alpha"] + df["mm_beta"] * df["ret_spy_0"])
df["ar_1_check"] = df["ret_1"] - (df["mm_alpha"] + df["mm_beta"] * df["ret_spy_1"])
df["car_check"]  = df["ar_0_check"] + df["ar_1_check"]

# ── 2. Check stored car_0_1 matches recomputed ────────────────────────────────
corr  = df["car_0_1"].corr(df["car_check"])
maxdiff = (df["car_0_1"] - df["car_check"]).abs().max()
print(f"Correlation stored vs recomputed : {corr:.6f}  (should be ~1.0)")
print(f"Max absolute difference          : {maxdiff:.2e}  (should be ~0)")

# ── 3. Check day 0 is now included ───────────────────────────────────────────
# If day 0 were missing, car_0_1 would equal ar_1 only
ar1_only = df["ret_1"] - (df["mm_alpha"] + df["mm_beta"] * df["ret_spy_1"])
corr_ar1 = df["car_0_1"].corr(ar1_only)
print(f"\nCorrelation with AR[+1] only     : {corr_ar1:.4f}  (should be < 0.95, not 1.0)")

# ── 4. Distribution check ─────────────────────────────────────────────────────
print(f"\nCAR[0,+1] distribution:")
print(df["car_0_1"].describe(percentiles=[.10, .25, .50, .75, .90]).round(4))

# ── 5. Downside label check ───────────────────────────────────────────────────
threshold = df["car_0_1"].quantile(0.25)
expected_downside = (df["car_0_1"] < threshold).sum()
print(f"\nDownside threshold (25th pct)    : {threshold:.4f} ({threshold*100:.2f}%)")
print(f"Stored downside count            : {df['downside'].sum()}")
print(f"Expected downside count (25th)   : {expected_downside}")
print(f"Match                            : {df['downside'].sum() == expected_downside}")

# ── 6. Spot check 3 known large drops ─────────────────────────────────────────
print(f"\nSpot check — known large drops:")
for ticker, date in [("DXCM", "2024-07-25"), ("CNC", "2025-07-01"), ("FMC", "2025-10-29")]:
    row = df[(df["ticker"] == ticker) & (df["filingDate"] == date)]
    if len(row):
        r = row.iloc[0]
        print(f"  {ticker} {date}: car_0_1={r['car_0_1']:.4f} | "
              f"ret_0={r['ret_0']:.4f} | ret_1={r['ret_1']:.4f} | downside={r['downside']}")

# Dataset Variables — filings_labeled.csv

## Identification
| Variable | Type | Description |
|---|---|---|
| `ticker` | string | Stock ticker symbol (e.g. AAPL, MSFT) |
| `cik` | string | SEC Central Index Key — unique company identifier in EDGAR |
| `accessionNumber` | string | SEC filing accession number — unique identifier for each 8-K filing |

## Filing Metadata
| Variable | Type | Description |
|---|---|---|
| `filingDate` | date | Calendar date the 8-K was filed with the SEC (YYYY-MM-DD) |
| `filingDatetime` | datetime | Exact filing timestamp in UTC, sourced from EDGAR's ACCEPTANCE-DATETIME field |

## Event Window Dates
| Variable | Type | Description |
|---|---|---|
| `event_date` | date | Day 0 — first trading day the market can react to the filing. Equal to `filingDate` if filed before 4pm ET, otherwise rolled to the next NYSE trading day |
| `event_date_m1` | date | Day −1 — the NYSE trading day immediately before `event_date`. Used for the CAR[−1,+1] robustness window |
| `event_date_p1` | date | Day +1 — the NYSE trading day immediately after `event_date`. Used in the primary CAR[0,+1] window |

## Stock Prices
| Variable | Type | Description |
|---|---|---|
| `price_m1` | float | Adjusted closing price of the stock on day −1 |
| `price_0` | float | Adjusted closing price of the stock on day 0 |
| `price_p1` | float | Adjusted closing price of the stock on day +1 |
| `spy_price_m1` | float | Adjusted closing price of SPY (S&P 500 ETF) on day −1. Used as market benchmark |
| `spy_price_0` | float | Adjusted closing price of SPY on day 0 |
| `spy_price_p1` | float | Adjusted closing price of SPY on day +1 |

## Derived Variables (computed from raw prices as needed)
These are not stored in the file but can be computed in one line:

| Variable | Formula | Description |
|---|---|---|
| `ret_stock` | `(price_p1 - price_0) / price_0` | Simple return of the stock over the [0,+1] window |
| `ret_spy` | `(spy_price_p1 - spy_price_0) / spy_price_0` | Simple return of SPY over the [0,+1] window |
| `car_0_1` | `ret_stock - ret_spy` | Cumulative abnormal return — primary outcome variable. Market-adjusted model |
| `downside` | `1 if car_0_1 < -0.02 else 0` | Binary label — 1 if the filing was followed by a significant negative market reaction |

## Notes
- Prices are **adjusted** for splits and dividends (sourced from Yahoo Finance via `yfinance`)
- The after-hours cutoff is **4pm ET** (NYSE close). Filings after this time are assigned to the next trading day
- NYSE holidays and weekends are handled via the `pandas-market-calendars` library
- The text content of each filing is stored separately in `filings_with_text.csv` and joined on `accessionNumber`